# LAB | Extractive Question Answering

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

Install Dependencies

In [1]:
!pip install -U langchain langchain-core langchain-classic langchain-pinecone langchain-huggingface datasets pinecone-client sentence-transformers torch


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\rache\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
pip install -U pinecone-client

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip uninstall -y pinecone-client pinecone

Found existing installation: pinecone-client 6.0.0
Uninstalling pinecone-client-6.0.0:
  Successfully uninstalled pinecone-client-6.0.0
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pinecone

  Using cached pinecone-8.1.0-py3-none-any.whl.metadata (14 kB)
  Using cached pinecone_plugin_assistant-3.0.2-py3-none-any.whl.metadata (30 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
Using cached pinecone-8.1.0-py3-none-any.whl (742 kB)
Using cached pinecone_plugin_assistant-3.0.2-py3-none-any.whl (280 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)

  Attempting uninstall: packaging

    Found existing installation: packaging 25.0

    Uninstalling packaging-25.0:

      Successfully uninstalled packaging-25.0

   -------------------- ------------------- 2/4 [pinecone-plugin-assistant]
   -------------------- ------------------- 2/4 [pinecone-plugin-assistant]
   ------------------------------ --------- 3/4 [pinecone]
   ------------------------------ --------- 3/4 [pinecone]
   ------------------------------ --------- 3/4 [pinecone]
   ------------------------------ --------- 3/4 [pinecone]
   ---------------------------------------- 4/4 [pinecone]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
from pinecone import Pinecone, ServerlessSpec
# Cela ne devrait plus lever d'Exception

In [5]:
import getpass
import os
from dotenv import load_dotenv
load_dotenv() # Charge la clé depuis le fichier .env

from pinecone import Pinecone, ServerlessSpec

# Prompt for API key if not set
if not os.getenv("PINECONE_API_KEY"):
    api_key_input = getpass.getpass("Enter your Pinecone API key: ")
    os.environ["PINECONE_API_KEY"] = api_key_input

pinecone_api_key = os.environ.get("PINECONE_API_KEY")

# Verify the key is set (print masked version)
if pinecone_api_key:
    print(f"Pinecone API Key (masked): {pinecone_api_key[:4]}...{pinecone_api_key[-4:]}")
else:
    print("Pinecone API Key is not set.")

Pinecone API Key (masked): pcsk...NjCZ


# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [7]:
pip install datasets pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
!pip install -q datasets pandas transformers sentence-transformers pinecone-client python-dotenv


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\rache\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [6]:
from datasets import load_dataset

# load the squad dataset into a pandas dataframe
df = load_dataset("squad", split="train").to_pandas()

c:\Users\rache\anaconda3\envs\lab_qa\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# select only title and context column
df = None
# drop rows containing duplicate context passages
df = None
df

# Initialize Pinecone Index

The Pinecone index stores vector representations of our context passages which we can retrieve using another vector (query vector). We first need to initialize our connection to Pinecone to create our vector index. For this, we need a free [API key]("https://app.pinecone.io/"), and then we initialize the connection like so:

In [8]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(api_key=pinecone_api_key, environment=spec.region)

Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [9]:
import time
from pinecone import ServerlessSpec

index_name = "extractive-question-answering"

# 1. Vérifier si l'index existe, sinon le créer
if index_name not in pc.list_indexes().names():
    print(f"Création de l'index '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=384, # Dimension standard pour le modèle 'multi-qa-MiniLM-L6-cos-v1' souvent utilisé dans ce lab
        metric='cosine',
        spec=ServerlessSpec(
            cloud='aws', 
            region='us-east-1' # Ou la région disponible sur votre compte gratuit
        )
    )
    # Attendre que l'index soit prêt
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)
    print("Index créé avec succès !")
else:
    print(f"L'index '{index_name}' existe déjà.")

# 2. Maintenant, la connexion fonctionnera
index = pc.Index(index_name)
print("Connecté à l'index.")

L'index 'extractive-question-answering' existe déjà.
Connecté à l'index.


# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [13]:
!pip install sentence-transformers torch langchain-huggingface


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\rache\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [14]:
pip install --upgrade --force-reinstall numpy

  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.


In [15]:
pip install --upgrade ml_dtypes

Note: you may need to restart the kernel to use updated packages.


In [16]:
# Installe toutes les dépendances critiques du LAB
!pip install -q sentence-transformers pinecone datasets pandas transformers python-dotenv


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\rache\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [17]:
# Installe toutes les dépendances critiques du LAB
!pip install -q sentence-transformers pinecone datasets pandas transformers python-dotenv


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: C:\Users\rache\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [10]:
import torch
from sentence_transformers import SentenceTransformer

# Set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load retriever model (multi-qa-MiniLM-L6-cos-v1 - 384 dimensions)
retriever = None #use the 'multi-qa-MiniLM-L6-cos-v1' model from HuggingFace to build the retriever

# ✅ Now retriever is loaded and ready
print("Retriever model loaded:", retriever)

Using device: cpu
Retriever model loaded: None


# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [11]:
from datasets import load_dataset
# Cette ligne crée la variable 'df'
df = load_dataset("squad", split="train").to_pandas()

# Ajoutez cette ligne pour vérifier que ça a marché :
print(f"Le dataset contient {len(df)} lignes.")

Le dataset contient 87599 lignes.


In [12]:
from sentence_transformers import SentenceTransformer

# Initialisation du modèle (le "retriever")
# On utilise souvent ce modèle pour la recherche de questions/réponses
model_name = 'multi-qa-MiniLM-L6-cos-v1'
retriever = SentenceTransformer(model_name)

# Déplacement sur GPU si disponible
if torch.cuda.is_available():
    retriever.to('cuda')

print("Retriever prêt !")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 675.80it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever prêt !


In [13]:
for i in tqdm(range(0, len(df), batch_size)):
    # 1. Définir la fin du lot
    i_end = min(i + batch_size, len(df))
    
    # 2. Extraire le lot
    batch = df.iloc[i:i_end]
    
    # 3. Générer les embeddings (bien s'assurer du .tolist())
    emb = retriever.encode(batch['context'].tolist()).tolist()
    
    # 4. PRÉPARER LES MÉTADONNÉES (C'est ici qu'on corrige !)
    # On ne garde que 'title' et 'context' pour éviter les erreurs de format
    meta = batch[['title', 'context']].to_dict(orient='records')
    
    # 5. Créer des IDs uniques
    ids = [f"{idx}" for idx in batch.index]
    
    # 6. Créer la liste pour l'upsert
    to_upsert = list(zip(ids, emb, meta))
    
    # 7. Envoyer vers Pinecone
    _ = index.upsert(vectors=to_upsert)

print("Indexation terminée !")
print(index.describe_index_stats())

NameError: name 'tqdm' is not defined

# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [14]:
from transformers import pipeline

model_name = 'deepset/electra-base-squad2'
# load the reader model into a question-answering pipeline
reader = pipeline(tokenizer=model_name, model=model_name, task='question-answering', device=device)
reader

c:\Users\rache\anaconda3\envs\lab_qa\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rache\.cache\huggingface\hub\models--deepset--electra-base-squad2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 631.76it/s, Materializing param=qa_outputs.weigh

QuestionAnsweringPipeline: {'model': 'ElectraForQuestionAnswering', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [15]:
# gets context passages from the pinecone index
def get_context(question, top_k):
    # generate embeddings for the question
    xq = None
    # search pinecone index for context passage with the answer
    xc = None
    # extract the context passage from pinecone search result
    c = None
    return c



In [16]:
from pprint import pprint

# extracts answer from the context passage
def extract_answer(question, context):
    results = []
    for c in context:
        # feed the reader the question and contexts to extract answers
        answer = reader(question=question, context=c)
        # add the context to answer dict for printing both together
        answer["context"] = c
        results.append(answer)
    # sort the result based on the score from reader model
    sorted_result = pprint(sorted(results, key=lambda x: x['score'], reverse=True))
    return sorted_result


In [17]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 3)
context

As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [24]:
extract_answer(question, context)

TypeError: 'NoneType' object is not iterable

The reader model predicted with 99% accuracy the correct answer *691,000 bbl/d* as seen from the context passage. Let's run few more queries.

In [25]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

TypeError: 'NoneType' object is not iterable

In [20]:
question = "What is Albert Eistein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

TypeError: 'NoneType' object is not iterable

Let's run another question. This time for top 3 context passages from the retriever.

In [21]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

TypeError: 'NoneType' object is not iterable

The result looks pretty good.

In [22]:
pc.delete_index(index_name)

### Add a few more questions. What did you observe?